# Stage 2 — Reproducing Levin & Yehudai's models

**Purpose.** Rebuild their classifiers from the published dataset and check the numbers against the
paper. If this reproduces, the pipeline is faithful and the CodeBERT comparison in Stage 4 means
something. If it does not, that has to be fixed before anything is built on top.

**ChangeDistiller is not needed.** The 48 change-type features are already in the dataset, so their
full method is trainable from the CSV alone.

### What is being reproduced

Three **feature sets**:

| model type | features |
|---|---|
| Keywords | 20 binary keyword indicators |
| Changes | 48 source-code change-type counts |
| Combined | all 68 |

And the **compound model**, which is their actual contribution. Commit messages often contain none
of the keywords — 28% to 34.8% by their measurement — and for those a keyword model has an all-zero
feature vector and no signal. So two models are held, and which one classifies a commit depends on
whether it has keywords at all:

```
classify(commit):
    if hasKeywords(commit.comment):  modelKW
    else:                            modelKW'
```

Each half may be any of the three types, giving nine combinations per algorithm.

### Targets

**Table 7** — test set, best compound model per algorithm:

| algorithm | modelKW | modelKW′ | accuracy | κ |
|---|---|---|---|---|
| J48 | Keywords | Changes | 70.3% | 53.9% |
| GBM | Combined | Combined | 72.6% | 57.2% |
| **RF** | **Keywords** | **Combined** | **76.7%** | **63.5%** |

**Table 8** gives the champion model's confusion matrix over 172 test commits, with per-class
recall of 75 / 84 / 68% and precision of 73 / 75 / 80%. That is a far stronger check than accuracy
alone — a model can hit 76% with quite different per-class behaviour.

## 0. Setup

Run this cell first. It works in Google Colab, Jupyter and IDEs such as VS Code or PyCharm.

It finds this repository (on Colab, opened straight from GitHub, it clones it), then picks one
working folder, `WORK`, where this notebook reads its inputs and writes its outputs. Every
notebook in the repository uses the same `WORK`, so they hand files to each other through it.

- set the environment variable `CIC_WORK` to choose the folder yourself
- otherwise it is `<repository>/work`, or on Colab a folder on your Google Drive
  (set `CIC_DRIVE=0` to stay on Colab's local disk)

An input missing from `WORK` is copied from the repository or downloaded from this study's Zenodo
record or Hugging Face repository, so any notebook can be run without re-running the ones before
it. See `replication.py`.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/Binamra00/commit-intent-and-refactoring-levels"
_candidates = [Path(os.environ["CIC_ROOT"])] if os.environ.get("CIC_ROOT") else []
_candidates += [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "replication.py").exists()), None)
if ROOT is None:                      # opened on its own, e.g. from GitHub in Colab
    ROOT = Path.cwd() / REPO.rsplit("/", 1)[-1]
    if not (ROOT / "replication.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))

from replication import setup, require
WORK = setup()

## 1. Load Stage 1's output

`levin_columns.json` holds the resolved column names, so nothing is re-derived here.

In [1]:
import json, collections
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, cohen_kappa_score, confusion_matrix,
                             classification_report)

need = ["levin_dataset.pkl", "levin_columns.json"]
missing = [f for f in need if not Path(f).exists()]
if missing:
    raise SystemExit(f"missing {missing} -- run Stage 1 first, in this directory")

D = pd.read_pickle("levin_dataset.pkl")
COLS = json.load(open("levin_columns.json"))

KW_COLS, CH_COLS = COLS["keywords"], COLS["changes"]
LABEL, ID, PROJECT, MESSAGE = COLS["label"], COLS["hash"], COLS["project"], COLS["message"]

print(f"{len(D):,} rows")
print(f"   keyword features  {len(KW_COLS)}")
print(f"   change features   {len(CH_COLS)}")
print(f"   combined          {len(KW_COLS) + len(CH_COLS)}")
print(f"   label column      {LABEL!r}")

NAMES = {"c": "corrective", "p": "perfective", "a": "adaptive"}
D["_y"] = D[LABEL].astype(str).str.strip().str.lower().map(NAMES)
print(f"\n{D['_y'].value_counts().to_string()}")

1,151 rows
   keyword features  20
   change features   48
   combined          68
   label column      'label'

_y
corrective    500
perfective    404
adaptive      247


### The duplicate commit

Stage 1 found one commit id appearing twice. Left alone it could land in both the training and test
halves, which inflates the score — the model would be scored on a commit it had already seen.

It is dropped rather than grouped. One row in 1,151 does not change any result, and dropping keeps
the split logic simple enough to trust.

In [2]:
dups = D[D[ID].duplicated(keep=False)].sort_values(ID)
if len(dups):
    print(f"{len(dups)} rows share {dups[ID].nunique()} commit id(s):\n")
    print(dups[[ID, PROJECT, LABEL]].to_string(index=False))
    D = D.drop_duplicates(subset=[ID], keep="first").reset_index(drop=True)
    print(f"\ndropped, leaving {len(D):,} rows")
else:
    print("no duplicate commit ids")

4 rows share 2 commit id(s):

                                commitId          project label
65a5e2cc46b2c04591aa59b7a85751479a7dbd0b           hadoop     a
65a5e2cc46b2c04591aa59b7a85751479a7dbd0b           hadoop     a
f26b7b2321ba561c947da92510e95e22f059351c ReactiveX-RxJava     c
f26b7b2321ba561c947da92510e95e22f059351c ReactiveX-RxJava     p

dropped, leaving 1,149 rows


## 2. Feature sets, and which commits have keywords

`hasKeywords` is what routes a commit to one half of the compound model or the other. A commit whose
20 keyword indicators are all zero contains none of the stems, and is what the second half exists
for.

In [3]:
X_KW  = D[KW_COLS].astype(float)
X_CH  = D[CH_COLS].astype(float)
X_ALL = pd.concat([X_KW, X_CH], axis=1)
y = D["_y"].to_numpy(dtype=object)     # a plain array on pandas 2 and 3 alike

FEATURES = {"Keywords": X_KW, "Changes": X_CH, "Combined": X_ALL}

HAS_KW = (X_KW.sum(axis=1) > 0).values
print(f"commits with at least one keyword : {HAS_KW.sum():,} ({HAS_KW.mean()*100:.1f}%)")
print(f"commits with none                 : {(~HAS_KW).sum():,} ({(~HAS_KW).mean()*100:.1f}%)")
print("\nThe paper reports 28% to 34.8% without keywords.")
print()
print("label distribution within each group:\n")
print(pd.crosstab(pd.Series(np.where(HAS_KW, "has keywords", "none"), name=""),
                  pd.Series(y, name="label"), normalize="index").mul(100).round(1).to_string())
print("\nIf the two groups have different label mixes, routing by keyword presence is not")
print("neutral -- the second half faces a different problem, not merely a harder one.")

commits with at least one keyword : 826 (71.9%)
commits with none                 : 323 (28.1%)

The paper reports 28% to 34.8% without keywords.

label distribution within each group:

label         adaptive  corrective  perfective
                                              
has keywords      23.1        46.1        30.8
none              17.0        36.8        46.1

If the two groups have different label mixes, routing by keyword presence is not
neutral -- the second half faces a different problem, not merely a harder one.


## 3. Split

Their protocol: a single 85/15 stratified split. The test set should land at about 172 commits,
which is the size of the confusion matrix in Table 8.

In [4]:
SEED = 42
idx = np.arange(len(D))
tr, te = train_test_split(idx, test_size=0.15, random_state=SEED, stratify=y)

print(f"train {len(tr):,}   test {len(te):,}")
print(f"paper's test set is 172 commits (table 8 totals) -> "
      f"{'comparable' if abs(len(te) - 172) <= 3 else 'DIFFERS'}")
print()
print(pd.DataFrame({
    "train": pd.Series(y[tr]).value_counts(),
    "test": pd.Series(y[te]).value_counts(),
}).assign(**{"test %": lambda d: (d["test"] / d["test"].sum() * 100).round(1)}).to_string())

train 976   test 173
paper's test set is 172 commits (table 8 totals) -> comparable

            train  test  test %
corrective    425    75    43.4
perfective    342    61    35.3
adaptive      209    37    21.4


## 4. Single models

Each feature set with each algorithm, before any compounding. This establishes the baselines the
compound models have to beat.

**On the algorithms.** The paper used R's caret with J48, GBM and RF. J48 is Weka's C4.5;
scikit-learn has no C4.5, so `DecisionTreeClassifier` (CART) stands in. It is a near equivalent,
not the same algorithm, and J48 results here should be read as approximate. GBM and RF map
directly.

In [5]:
def make(alg, seed=SEED):
    if alg == "J48":     # C4.5 has no scikit-learn equivalent; CART is the near neighbour
        return DecisionTreeClassifier(random_state=seed)
    if alg == "GBM":
        return GradientBoostingClassifier(random_state=seed)
    if alg == "RF":
        return RandomForestClassifier(n_estimators=500, random_state=seed, n_jobs=-1)
    raise ValueError(alg)


ALGS = ["J48", "GBM", "RF"]

rows = []
SINGLE = {}
for alg in ALGS:
    for name, X in FEATURES.items():
        m = make(alg).fit(X.iloc[tr], y[tr])
        p = m.predict(X.iloc[te])
        SINGLE[(alg, name)] = m
        rows.append({"algorithm": alg, "features": name,
                     "accuracy": round(accuracy_score(y[te], p) * 100, 1),
                     "kappa": round(cohen_kappa_score(y[te], p) * 100, 1)})

S = pd.DataFrame(rows)
print("SINGLE MODELS, test set\n")
print(S.pivot(index="features", columns="algorithm", values="accuracy")
      .reindex(["Keywords", "Changes", "Combined"]).to_string())
print("\nkappa\n")
print(S.pivot(index="features", columns="algorithm", values="kappa")
      .reindex(["Keywords", "Changes", "Combined"]).to_string())
print("""
The paper observes that models relying only on commit messages beat those relying only on source
code changes, under all three algorithms. Keywords should therefore outscore Changes here.""")

SINGLE MODELS, test set

algorithm   GBM   J48    RF
features                   
Keywords   72.8  73.4  74.0
Changes    53.2  46.8  49.7
Combined   78.0  67.6  75.7

kappa

algorithm   GBM   J48    RF
features                   
Keywords   58.0  59.0  59.9
Changes    24.3  16.5  18.8
Combined   65.4  49.0  61.5

The paper observes that models relying only on commit messages beat those relying only on source
code changes, under all three algorithms. Keywords should therefore outscore Changes here.


## 5. Compound models

Two components, selected at prediction time by whether the commit has any keyword.

**One design choice that the paper leaves ambiguous.** It describes precomputing *"two models; one
to be used when all features are present, and the other when only a subset is available"* — which
reads as two models over different **feature sets**, both trained on the full training data, with
routing at prediction time. The alternative reading is that each half is trained only on its own
subset of rows.

The first is implemented here, matching the pseudo-code. The second is computed alongside so the
difference is visible rather than assumed.

In [6]:
class Compound:
    """
    Two trained components; which one predicts a row depends on whether it has keywords.

    `per_subset=False` trains both components on all training rows, differing only in feature set.
    `per_subset=True` trains each on its own group of rows.
    """

    def __init__(self, alg, feats_kw, feats_nokw, per_subset=False):
        self.alg, self.fkw, self.fno, self.per_subset = alg, feats_kw, feats_nokw, per_subset

    def fit(self, tr_idx):
        kw_rows = tr_idx[HAS_KW[tr_idx]] if self.per_subset else tr_idx
        no_rows = tr_idx[~HAS_KW[tr_idx]] if self.per_subset else tr_idx
        self.m_kw = self.m_no = None
        if len(np.unique(y[kw_rows])) > 1:
            self.m_kw = make(self.alg).fit(FEATURES[self.fkw].iloc[kw_rows], y[kw_rows])
        if len(np.unique(y[no_rows])) > 1:
            self.m_no = make(self.alg).fit(FEATURES[self.fno].iloc[no_rows], y[no_rows])
        self.fallback = pd.Series(y[tr_idx]).mode()[0]
        return self

    def predict(self, te_idx):
        out = np.empty(len(te_idx), dtype=object)
        mask = HAS_KW[te_idx]
        for sel, model, feats in ((mask, self.m_kw, self.fkw),
                                  (~mask, self.m_no, self.fno)):
            if not sel.any():
                continue
            rows = te_idx[sel]
            out[sel] = (model.predict(FEATURES[feats].iloc[rows]) if model is not None
                        else self.fallback)
        return out


TYPES = ["Keywords", "Changes", "Combined"]
rows = []
for alg in ALGS:
    for fkw in TYPES:
        for fno in TYPES:
            for per_subset in (False, True):
                p = Compound(alg, fkw, fno, per_subset).fit(tr).predict(te)
                rows.append({"algorithm": alg, "modelKW": fkw, "modelKW'": fno,
                             "training": "per subset" if per_subset else "full",
                             "accuracy": round(accuracy_score(y[te], p) * 100, 1),
                             "kappa": round(cohen_kappa_score(y[te], p) * 100, 1)})
C = pd.DataFrame(rows)

print("COMPOUND MODELS, test set, components trained on the full training data\n")
full = C[C.training == "full"]
for alg in ALGS:
    sub = full[full.algorithm == alg].sort_values("kappa", ascending=False)
    print(f"{alg}")
    print(sub[["modelKW", "modelKW'", "accuracy", "kappa"]].to_string(index=False))
    print()

COMPOUND MODELS, test set, components trained on the full training data

J48
 modelKW modelKW'  accuracy  kappa
Keywords Keywords      73.4   59.0
Keywords Combined      71.1   55.0
Keywords  Changes      70.5   54.0
Combined Keywords      69.9   53.2
Combined Combined      67.6   49.0
Combined  Changes      67.1   48.1
 Changes Keywords      49.7   22.0
 Changes Combined      47.4   17.6
 Changes  Changes      46.8   16.5

GBM
 modelKW modelKW'  accuracy  kappa
Combined Combined      78.0   65.4
Keywords Combined      75.7   62.0
Combined Keywords      75.1   61.3
Combined  Changes      75.1   60.7
Keywords Keywords      72.8   58.0
Keywords  Changes      72.8   57.4
 Changes Combined      56.1   29.3
 Changes Keywords      53.2   25.5
 Changes  Changes      53.2   24.3

RF
 modelKW modelKW'  accuracy  kappa
Keywords Combined      75.7   62.1
Combined Combined      75.7   61.5
Keywords  Changes      75.1   61.1
Combined  Changes      75.1   60.4
Keywords Keywords      74.0   59.9
Comb

In [7]:
PAPER_T7 = {"J48": ("Keywords", "Changes", 70.3, 53.9),
            "GBM": ("Combined", "Combined", 72.6, 57.2),
            "RF":  ("Keywords", "Combined", 76.7, 63.5)}

print("AGAINST TABLE 7\n")
print(f"{'alg':<6}{'modelKW':<11}{'modelKW2':<11}{'ours acc':>10}{'paper':>8}"
      f"{'ours k':>9}{'paper':>8}")
print("-" * 65)
for alg, (fkw, fno, acc_p, kap_p) in PAPER_T7.items():
    r = full[(full.algorithm == alg) & (full["modelKW"] == fkw) & (full["modelKW'"] == fno)]
    if r.empty:
        continue
    a, k = r.iloc[0]["accuracy"], r.iloc[0]["kappa"]
    print(f"{alg:<6}{fkw:<11}{fno:<11}{a:>9.1f}%{acc_p:>7.1f}%{k:>8.1f}%{kap_p:>7.1f}%")

print("\n\nBEST BY KAPPA, ours vs the paper's choice\n")
for alg in ALGS:
    sub = full[full.algorithm == alg].sort_values("kappa", ascending=False).iloc[0]
    fkw_p, fno_p, _, _ = PAPER_T7[alg]
    same = (sub["modelKW"] == fkw_p) and (sub["modelKW'"] == fno_p)
    prime = sub["modelKW'"]
    verdict = "same combination" if same else "different combination"
    print(f"   {alg:<5}ours: {sub['modelKW']:<9}/ {prime:<9}k={sub['kappa']:.1f}%"
          f"    paper: {fkw_p} / {fno_p}   {verdict}")

AGAINST TABLE 7

alg   modelKW    modelKW2     ours acc   paper   ours k   paper
-----------------------------------------------------------------
J48   Keywords   Changes         70.5%   70.3%    54.0%   53.9%
GBM   Combined   Combined        78.0%   72.6%    65.4%   57.2%
RF    Keywords   Combined        75.7%   76.7%    62.1%   63.5%


BEST BY KAPPA, ours vs the paper's choice

   J48  ours: Keywords / Keywords k=59.0%    paper: Keywords / Changes   different combination
   GBM  ours: Combined / Combined k=65.4%    paper: Combined / Combined   same combination
   RF   ours: Keywords / Combined k=62.1%    paper: Keywords / Combined   same combination


### Does training each half on its own rows help?

The ambiguity from above, measured.

In [8]:
piv = C.pivot_table(index=["algorithm", "modelKW", "modelKW'"],
                    columns="training", values="kappa")
piv["difference"] = (piv["per subset"] - piv["full"]).round(1)
print("kappa under both readings\n")
print(piv.sort_values("difference").to_string())
print(f"\nmedian difference: {piv['difference'].median():+.1f} points")
print("If small, the ambiguity does not matter and the simpler reading stands.")

kappa under both readings

training                     full  per subset  difference
algorithm modelKW  modelKW'                              
GBM       Combined Combined  65.4        60.5        -4.9
                   Keywords  61.3        56.7        -4.6
RF        Combined Combined  61.5        57.9        -3.6
                   Changes   60.4        57.0        -3.4
J48       Combined Keywords  53.2        50.6        -2.6
RF        Combined Keywords  59.2        57.7        -1.5
          Keywords Changes   61.1        60.0        -1.1
                   Combined  62.1        61.0        -1.1
J48       Combined Combined  49.0        48.1        -0.9
GBM       Combined Changes   60.7        60.5        -0.2
J48       Keywords Keywords  59.0        59.0         0.0
          Combined Changes   48.1        48.1         0.0
GBM       Keywords Combined  62.0        62.7         0.7
RF        Keywords Keywords  59.9        60.7         0.8
GBM       Keywords Keywords  58.0        58.9

## 6. The champion model in detail

Table 8 gives the paper's confusion matrix for RF with modelKW=Keywords and modelKW′=Combined.
Comparing per-class recall and precision is a much stronger check than accuracy: two models can
score the same overall while behaving quite differently per class.

In [9]:
champ = Compound("RF", "Keywords", "Combined").fit(tr)
pred = champ.predict(te)
labels = ["adaptive", "corrective", "perfective"]

cm = confusion_matrix(y[te], pred, labels=labels)
ours = pd.DataFrame(cm.T, index=[f"classified {l}" for l in labels],
                    columns=[f"true {l}" for l in labels])
print("OURS\n")
print(ours.to_string())

paper = pd.DataFrame([[28, 5, 5], [6, 63, 14], [3, 7, 41]],
                     index=[f"classified {l}" for l in labels],
                     columns=[f"true {l}" for l in labels])
print("\n\nPAPER, table 8\n")
print(paper.to_string())

rec_o = cm.diagonal() / cm.sum(axis=1) * 100
pre_o = cm.diagonal() / cm.sum(axis=0) * 100
print("\n\nPER CLASS\n")
print(pd.DataFrame({
    "recall ours": rec_o.round(0), "recall paper": [75, 84, 68],
    "precision ours": pre_o.round(0), "precision paper": [73, 75, 80],
}, index=labels).to_string())

acc, kap = accuracy_score(y[te], pred) * 100, cohen_kappa_score(y[te], pred) * 100
nir = pd.Series(y[te]).value_counts(normalize=True).max() * 100
print(f"\naccuracy {acc:.1f}%   (paper 76%)")
print(f"kappa    {kap:.1f}%   (paper 63%)")
print(f"NIR      {nir:.1f}%   (paper 43%)")
print("\nNIR is the accuracy of always predicting the majority class. A model at or below it")
print("has learned nothing, whatever its accuracy looks like.")

OURS

                       true adaptive  true corrective  true perfective
classified adaptive               26                5                9
classified corrective              5               65               12
classified perfective              6                5               40


PAPER, table 8

                       true adaptive  true corrective  true perfective
classified adaptive               28                5                5
classified corrective              6               63               14
classified perfective              3                7               41


PER CLASS

            recall ours  recall paper  precision ours  precision paper
adaptive           70.0            75            65.0               73
corrective         87.0            84            79.0               75
perfective         66.0            68            78.0               80

accuracy 75.7%   (paper 76%)
kappa    62.1%   (paper 63%)
NIR      43.4%   (paper 43%)

NIR is the accuracy of

## 7. Stability

The 85/15 split is a single draw. Cross-validation says whether the headline number is a property
of the model or of that particular split. Reported for information; the split above is what Stage 4
compares against, since it is the paper's protocol.

In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rows = []
for alg in ALGS:
    for name, X in FEATURES.items():
        p = cross_val_predict(make(alg), X, y, cv=cv, n_jobs=-1)
        rows.append({"algorithm": alg, "features": name,
                     "accuracy": round(accuracy_score(y, p) * 100, 1),
                     "kappa": round(cohen_kappa_score(y, p) * 100, 1)})
CV = pd.DataFrame(rows)
print("5-FOLD CROSS-VALIDATION, single models\n")
print(CV.pivot(index="features", columns="algorithm", values="accuracy")
      .reindex(["Keywords", "Changes", "Combined"]).to_string())

comp = S.merge(CV, on=["algorithm", "features"], suffixes=("_split", "_cv"))
comp["gap"] = (comp.accuracy_split - comp.accuracy_cv).round(1)
print("\n\nsingle 85/15 split minus cross-validated\n")
print(comp[["algorithm", "features", "accuracy_split", "accuracy_cv", "gap"]].to_string(index=False))
worst = comp.gap.abs().max()
print(f"\nlargest gap: {worst:.1f} points")
if worst > 8:
    print("Large. The single split is an unstable estimate -- report it as one draw, not a point")
    print("estimate, and prefer the cross-validated figure when comparing against Stage 4.")

5-FOLD CROSS-VALIDATION, single models

algorithm   GBM   J48    RF
features                   
Keywords   71.2  68.8  71.0
Changes    53.0  47.8  52.9
Combined   72.8  64.1  72.0


single 85/15 split minus cross-validated

algorithm features  accuracy_split  accuracy_cv  gap
      J48 Keywords            73.4         68.8  4.6
      J48  Changes            46.8         47.8 -1.0
      J48 Combined            67.6         64.1  3.5
      GBM Keywords            72.8         71.2  1.6
      GBM  Changes            53.2         53.0  0.2
      GBM Combined            78.0         72.8  5.2
       RF Keywords            74.0         71.0  3.0
       RF  Changes            49.7         52.9 -3.2
       RF Combined            75.7         72.0  3.7

largest gap: 5.2 points


## 8. Save

The champion configuration and the split, so Stage 4 compares CodeBERT against the same rows rather
than a fresh partition.

In [11]:
np.save("stage2_train_idx.npy", tr)
np.save("stage2_test_idx.npy", te)
C.to_csv("stage2_compound_results.csv", index=False)
S.to_csv("stage2_single_results.csv", index=False)

best = full.sort_values("kappa", ascending=False).iloc[0]
summary = {
    "n_rows": int(len(D)),
    "n_train": int(len(tr)), "n_test": int(len(te)),
    "seed": SEED,
    "best": {"algorithm": best["algorithm"], "modelKW": best["modelKW"],
             "modelKW_prime": best["modelKW'"],
             "accuracy": float(best["accuracy"]), "kappa": float(best["kappa"])},
    "paper_table7_rf": {"accuracy": 76.7, "kappa": 63.5},
    "champion_as_paper": {"accuracy": round(acc, 1), "kappa": round(kap, 1)},
}
json.dump(summary, open("stage2_summary.json", "w"), indent=1)

print(json.dumps(summary, indent=1))
print("\n   stage2_train_idx.npy, stage2_test_idx.npy, stage2_*_results.csv,")
print("   stage2_summary.json written")
print("""
Stage 4 must reuse these split indices. A CodeBERT model evaluated on a different partition is not
comparable to the numbers above, however close the figures happen to look.""")

{
 "n_rows": 1149,
 "n_train": 976,
 "n_test": 173,
 "seed": 42,
 "best": {
  "algorithm": "GBM",
  "modelKW": "Combined",
  "modelKW_prime": "Combined",
  "accuracy": 78.0,
  "kappa": 65.4
 },
 "paper_table7_rf": {
  "accuracy": 76.7,
  "kappa": 63.5
 },
 "champion_as_paper": {
  "accuracy": 75.7,
  "kappa": 62.1
 }
}

   stage2_train_idx.npy, stage2_test_idx.npy, stage2_*_results.csv,
   stage2_summary.json written

Stage 4 must reuse these split indices. A CodeBERT model evaluated on a different partition is not
comparable to the numbers above, however close the figures happen to look.
